In [2]:
# 1. Install required libraries (Safest Kaggle install for Unsloth)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

import os
# SAFEGUARD 1: Force Unsloth to use only 1 GPU on Kaggle's dual-T4 setup to prevent DDP crashes
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel, get_chat_template
from unsloth.chat_templates import standardize_sharegpt
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from kaggle_secrets import UserSecretsClient
import torch

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ncggvust/unsloth_5b01bffcbd18492c885dd4a1631c588f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ncggvust/unsloth_5b01bffcbd18492c885dd4a1631c588f
  Resolved https://github.com/unslothai/unsloth.git to commit cc9128134156cc51f94eeb76d3d19f80bddbc8a7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# 2. Authenticate with Hugging Face
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

In [5]:
# 3. Load Model and Tokenizer
max_seq_length = 4096
dtype = None # Auto-detects float16 for Kaggle T4
load_in_4bit = True 

# Use the official IBM Granite 3.1 model. Unsloth will auto-quantize it.
model_id = "ibm-granite/granite-3.1-2b-instruct"

print(f"Loading {model_id} in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

Loading ibm-granite/granite-3.1-2b-instruct in 4-bit...
==((====))==  Unsloth 2026.10.3: Fast Granite patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Unsloth: ibm-granite/granite-3.1-2b-instruct had a bad pad_token (<|end_of_text|>). Using pad_token = <fim_pad>.


In [6]:
# 4. Configure LoRA (Low-Rank Adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", # SAFEGUARD 2: Unsloth's custom checkpointing saves ~30% VRAM
    random_state = 42,
)


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [9]:
# 5. Load and Merge Phase 1 Datasets
data_dir = "/kaggle/input/datasets/siddardhashayini3/granite-claude-data/"

print("Loading Phase 1 Datasets...")
ds_prog = load_dataset("json", data_files=data_dir + "01_programming.jsonl", split="train")
ds_arch = load_dataset("json", data_files=data_dir + "06_architecture.jsonl", split="train")
ds_test = load_dataset("json", data_files=data_dir + "07_testing.jsonl", split="train")

# Combine, shuffle, and explicitly filter out any corrupted/empty rows
phase1_dataset = concatenate_datasets([ds_prog, ds_arch, ds_test]).shuffle(seed=42)
phase1_dataset = phase1_dataset.filter(lambda x: len(x["conversations"]) > 0)
print(f"Total Phase 1 Examples: {len(phase1_dataset)}")

Loading Phase 1 Datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/18000 [00:00<?, ? examples/s]

Total Phase 1 Examples: 18000


In [10]:
# 6. Apply ShareGPT Chat Template using Unsloth's native tool
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

# SAFEGUARD 3: This officially maps 'human' -> 'user' and 'gpt' -> 'assistant'
phase1_dataset = standardize_sharegpt(phase1_dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

formatted_dataset = phase1_dataset.map(formatting_prompts_func, batched = True)


[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_f3ppswuf/tokenizer_config.json.


Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

In [11]:
# 7. Configure the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # SAFEGUARD 4: Prevents OOM spikes at 4K context
        gradient_accumulation_steps = 8, # 1 * 8 = Effective Batch size of 8
        warmup_steps = 50,
        max_steps = 1000, # 1000 steps is a good starting point for Phase 1. 
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        save_strategy = "no", # SAFEGUARD 5: Prevents Kaggle "Disk Space Full" crashes.
        report_to = "none", # Disables wandb so it doesn't ask for API keys
    ),
)

# Clear PyTorch memory cache before starting
torch.cuda.empty_cache()

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/18000 [00:00<?, ? examples/s]

In [12]:
# 8. Run Training
print("Starting Phase 1 Training...")
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 49155}.


Starting Phase 1 Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 28,180,480 of 2,561,712,128 (1.10% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.088187
20,0.823181
30,0.689352
40,0.723598
50,0.839606
60,0.774356
70,0.649145
80,0.686641
90,0.630605
100,0.677999


In [13]:
# 9. Push to Hugging Face
your_hf_username = "SiddardhaShayini"
repo_name = f"{your_hf_username}/Granite-Claude-Code-Phase-1"

print(f"Pushing LoRA adapters to {repo_name}...")
# SAFEGUARD 6: Pushing directly to HF means your weights survive even if Kaggle disconnects right after.
model.push_to_hub(repo_name, token = hf_token)
tokenizer.push_to_hub(repo_name, token = hf_token)

print("✅ Phase 1 Complete and Saved!")


Pushing LoRA adapters to SiddardhaShayini/Granite-Claude-Code-Phase-1...


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpo3kdj3f_/tokenizer_config.json.


Saved model to https://huggingface.co/SiddardhaShayini/Granite-Claude-Code-Phase-1
✅ Phase 1 Complete and Saved!
